# Model Router Toolkit — Getting Started

LLM routing toolkit that learns which model handles which queries best, then routes each query to the **cheapest model above an accuracy threshold**. This notebook walks through the full setup using the `model-router` CLI: install, train, evaluate, and serve.

The router uses a **prefill-based** approach — a single forward pass through a lightweight encoder (Qwen3.5-35B-A3B) extracts hidden-state features, and a trained MLP ensemble predicts P(correct) per target model.

**Prerequisites:** Python 3.10+, ~20 GB disk for encoder model (downloaded from HuggingFace on first use).

---
## 1. Install

In [ ]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print(f"Working directory: {os.getcwd()}")

In [ ]:
%pip install -q -e '.[prefill,litellm]'

---
## 2. Train the Router

The reproduce script runs `model-router train` with pre-extracted prefill features. **Lean mode** finishes in ~30 seconds on CPU.

The v1 dataset covers three benchmarks — MMLU Pro, LiveCodeBench, and Humanity's Last Exam — evaluated across a 9-model pool ranging from Nemotron 3 Nano ($0.05/M input) to Claude Opus 4.6 ($2.77/M input).

| Required file | Description |
|---------------|-------------|
| `data/train_v1.csv` | Training labels (9 models) |
| `data/test_v1.csv` | Test labels (9 models) |
| `data/v1-9models-lean/train_features.pt` | Pre-transformed training features (~69 MB) |
| `data/v1-9models-lean/test_features.pt` | Pre-transformed test features (~20 MB) |

> Skip this cell if you already have `checkpoints/prefill_router_qwen35b.pt`.

In [ ]:
import os
if os.path.exists("checkpoints/prefill_router_qwen35b.pt"):
    sz = os.path.getsize("checkpoints/prefill_router_qwen35b.pt") / 1e6
    print(f"Checkpoint already exists ({sz:.1f} MB). Delete it and re-run to retrain.")
else:
    print("No checkpoint found — training from scratch...")
    !bash scripts/reproduce_v1_checkpoint.sh --lean

---
## 3. Evaluate the Checkpoint

The reproduce script already runs evaluation, but you can re-run it anytime with `model-router evaluate`.
This reports per-model AUC/accuracy, routing distribution, oracle vs router accuracy, and cost savings.

In [ ]:
!model-router evaluate \
    --config configs/v1-9models-qwen35b.yaml \
    --checkpoint checkpoints/prefill_router_qwen35b.pt \
    --data data/test_v1.csv \
    --features-from data/v1-9models-lean/test_features.pt \
    --device cpu

---
## 4. Serve the Router

Start the full server — it routes **and** calls the selected model via OpenRouter. The server exposes:
- OpenAI-compatible API at `/v1/chat/completions`
- Playground UI at `http://localhost:8000/`

You'll need an [OpenRouter API key](https://openrouter.ai/keys).

In [ ]:
import os
if not os.environ.get("OPENROUTER_API_KEY"):
    api_key = input("Enter your OpenRouter API key (https://openrouter.ai/keys): ")
    os.environ["OPENROUTER_API_KEY"] = api_key
print(f"OPENROUTER_API_KEY set ({len(os.environ['OPENROUTER_API_KEY'])} chars)")

In [ ]:
!model-router serve \
    --config configs/v1-9models-qwen35b.yaml \
    --port 8000 &

import time, requests
print("Waiting for server to load encoder...")
for attempt in range(30):
    time.sleep(5)
    try:
        resp = requests.get("http://localhost:8000/health", timeout=3)
        if resp.ok:
            print(f"Server ready after {(attempt + 1) * 5}s")
            print(f"  API:        http://localhost:8000/v1/chat/completions")
            print(f"  Playground: http://localhost:8000/")
            break
    except requests.ConnectionError:
        pass
else:
    print("Server did not start within 150s.")

### Call the API

The server is OpenAI-compatible. Send `"model": "routed"` and the router picks the best model for each query.

In [ ]:
!curl -s http://localhost:8000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{"model": "routed", "messages": [{"role": "user", "content": "What is the capital of France?"}]}' \
  | python -m json.tool

In [ ]:
!curl -s http://localhost:8000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{"model": "routed", "messages": [{"role": "user", "content": "Prove that the square root of 2 is irrational"}]}' \
  | python -m json.tool

---
## Cleanup

In [ ]:
!kill $(lsof -ti :8000) 2>/dev/null && echo "Server stopped." || echo "No server running on port 8000."

---
## What's Next

| Topic | Doc |
|-------|-----|
| Full config reference | [Configuration](../docs/configuration.md) |
| All integration paths | [Integration Guide](../docs/integration.md) |
| Adapters (LiteLLM, HTTP sidecar) | [Adapters Guide](../docs/adapters.md) |
| Gateway plugins (OpenClaw) | [Plugins Guide](../docs/plugins.md) |
| Collect, train, evaluate workflow | [Training Guide](../docs/training-guide.md) |
| Architecture and inference flow | [Architecture](../docs/architecture.md) |